# MM9 lab: security (30 min)

**What changed since MM8.** The network now has an eavesdropper. **Eve** can read and change everything on the link; **Mallory** is a router in the middle that can also replace what she forwards. Two questions, each anchored in Problem Set 5:

| Question | Time | | Problem Set 5 |
|---|---|---|---|
| 1 | 15 min | ECB vs. CBC: what does Eve learn, and how do errors spread? | Problem 1 |
| 2 | 15 min | Diffie-Hellman and the man in the middle | Problems 3, 4 and 6 |

**What you build:** four small functions in `student/mm9_security.py`, with Copilot. The block cipher, Diffie-Hellman and the chat simulator are given.

**Rhythm:** predict → run → observe → answer and check with `check_answer("MM9-1a", your_answer)`, which only says *correct* or *not correct*.

**Rules.** Prediction first, then run. Copilot writes code in `student/` only.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.insert(0, "..")   # the CommSim folder
from commsim import *
from commsim.checks import check_mm9_cbc, check_mm9_auth
from student.mm9_security import cbc_encrypt, cbc_decrypt, respond, verify

## Question 1: ECB vs. CBC (15 min)

> **Question.** What does Eve learn from ECB and from CBC ciphertext without the key, and how do errors spread when a block is damaged?

The lab uses a **toy block cipher** on 8-byte blocks, `toy_encrypt_block(block, key)`: a 4-round Feistel network. It behaves like a real block cipher for this lab (same block and key, same output; one changed bit scrambles the whole block), but it is **not secure**. Never use it for real data.

- **ECB** (given, `ecb_encrypt`): every block is encrypted on its own.
- **CBC** (you build it): every plaintext block is XORed with the previous ciphertext block before it is encrypted; the first one with an initialisation vector (IV), which is sent in the clear.

**Your task.** Ask Copilot to implement `cbc_encrypt` and `cbc_decrypt` in `student/mm9_security.py` from the specification, then run the check.

In [ ]:
check_mm9_cbc(cbc_encrypt, cbc_decrypt)

**1a. What Eve sees.** A temperature sensor sends a log of 64 blocks, but there are only 3 different readings (normal, warm, alarm).

*Predict first:* how many **different** ciphertext blocks does Eve see with ECB, and with CBC?

> **Your prediction:** *(write here, before you run the next cell)*

In [ ]:
log = sensor_log()
key, iv = DEMO_KEY, DEMO_IV
c_ecb = ecb_encrypt(log, key)
c_cbc = cbc_encrypt(log, key, iv)
print("different plaintext blocks: ", len(set(log)))
print("different ECB blocks:       ", len(set(c_ecb)))
print("different CBC blocks:       ", len(set(c_cbc)))
block_grid({"plaintext": log, "ECB (what Eve sees)": c_ecb, "CBC (what Eve sees)": c_cbc});

In [ ]:
check_answer("MM9-1a", ...)   # [different blocks with ECB, with CBC]

**1b. Damaged blocks.** On the way, one ciphertext block (number 20) gets one **flipped bit**, or is **lost** completely (the receiver then decrypts the remaining blocks in order).

*Predict first:* how many plaintext blocks are damaged with ECB, and with CBC, in each case?

> **Your prediction:** *(write here, before you run the next cell)*

In [ ]:
flipped = {"ECB": damaged_blocks(log, ecb_decrypt(flip_bit(c_ecb, block=20, bit=5), key)),
           "CBC": damaged_blocks(log, cbc_decrypt(flip_bit(c_cbc, block=20, bit=5), key, iv))}
lost = {"ECB": damaged_blocks(log, ecb_decrypt(drop_block(c_ecb, block=20), key), lost=20),
        "CBC": damaged_blocks(log, cbc_decrypt(drop_block(c_cbc, block=20), key, iv), lost=20)}
print("bit flip, damaged blocks:", flipped)
print("lost,     damaged blocks:", lost)
damage_plot({"ECB, bit flip": flipped["ECB"], "CBC, bit flip": flipped["CBC"],
             "ECB, lost block": lost["ECB"], "CBC, lost block": lost["CBC"]}, n_blocks=64);

In [ ]:
check_answer("MM9-1b", ...)   # [damaged blocks with ECB, with CBC]; check it for the flip and for the lost block

Problem Set 5's solution says that with CBC a lost block means "we cannot decrypt the following blocks". Does the simulation agree? Explain from the CBC decryption formula.

> **Your answer:** *(write here)*

**1c. Eve changes a value without the key.** Zoya's smart meter sends one 8-byte block with CBC: a 4-digit reading and a CRC-32 of those digits. The IV travels in the clear with it. Eve guesses that the reading is `"0100"` and wants Yoshi to read `"0900"`. She does not know the key, but she can flip bits in the IV (`eve_change`).

*Predict first:* does Yoshi's CBC decryption notice? Does the CRC catch it (first when Eve leaves the CRC alone, then when she also fixes it)?

> **Your prediction:** *(write here, before you run the next cell)*

In [ ]:
message = meter_message("0100")
c_meter = cbc_encrypt(message, key, iv)
for fix_crc in (False, True):
    iv_eve, c_eve = eve_change(iv, c_meter, old="0100", new="0900", fix_crc=fix_crc)
    reading, crc_ok = read_meter(cbc_decrypt(c_eve, key, iv_eve))
    print(f"Eve fixes the CRC: {fix_crc!s:<5}  Yoshi reads {reading}, CRC check passes: {crc_ok}")

In [ ]:
check_answer("MM9-1c", ...)   # [does CBC decryption notice?, would an unkeyed CRC notice?] (yes / no)

**In your own words (2 to 3 sentences):** what does encryption protect, and what does it not? What would Yoshi need to detect Eve's change?

> **Your answer:** *(write here)*

## Question 2: Diffie-Hellman and the man in the middle (15 min)

> **Question.** When can a router in the middle read a Diffie-Hellman-protected chat without anyone noticing, and what stops it?

Diffie-Hellman (Lecture 9): Zoya picks a secret *a* and sends *A* = *g*^*a* mod *p*; Yoshi picks *b* and sends *B* = *g*^*b* mod *p*. Both compute the same key: *B*^*a* = *A*^*b* mod *p*. Given: `dh_public(g, secret, p)` and `dh_shared(other_public, secret, p)`.

**2a. By hand first, then check.** *p* = 23, *g* = 5, *a* = 6, *b* = 15. What are Zoya's public value, Yoshi's public value and the shared key? Work it out on paper, then run the cell.

In [ ]:
p, g, a, b = 23, 5, 6, 15
A = dh_public(g, a, p)
B = dh_public(g, b, p)
print("Zoya's public value A =", A)
print("Yoshi's public value B =", B)
print("shared key:", dh_shared(B, a, p), "(Zoya)", dh_shared(A, b, p), "(Yoshi)")

In [ ]:
check_answer("MM9-2a", ...)   # [Zoya's public value, Yoshi's public value, shared key]

*Problem 6, for real:* the same two functions with the 1536-bit numbers from Problem Set 5. Compare the first digits with your own program.

In [ ]:
P6 = PS5_PROBLEM6
alpha, beta = dh_public(P6["g"], P6["a"], P6["p"]), dh_public(P6["g"], P6["b"], P6["p"])
print("alpha starts with", str(alpha)[:20], "...", len(str(alpha)), "digits")
print("beta  starts with", str(beta)[:20], "...")
print("same key:", dh_shared(beta, P6["a"], P6["p"]) == dh_shared(alpha, P6["b"], P6["p"]))

**2b. Mallory in the middle.** Zoya and Yoshi agree on a key over a router, then send 10 chat messages encrypted with it. Now the router is **Mallory**: she replaces both public values with her own, and relays and re-encrypts every message unchanged.

*Predict first:* do Zoya and Yoshi end up with the same key? How many of the 10 messages can Mallory read? Would they notice?

> **Your prediction:** *(write here, before you run the next cell)*

In [ ]:
honest = dh_chat(mallory=False)
attacked = dh_chat(mallory=True)
print(honest, "\n")
print(attacked)
chat_diagram(attacked);

In [ ]:
check_answer("MM9-2b", ...)   # [same key? (yes / no), messages Mallory read]

**2c. Authentication with a pre-shared key (Problem 4).** Zoya and Yoshi share a secret key *K* that Mallory does not know. Each sends a random challenge with its public value, and answers the other's challenge with a SHA-256 hash keyed with *K* that is **bound to its own public value**. Each side checks the answer against the public value it actually received.

**Your task.** Ask Copilot to implement `respond` and `verify` in `student/mm9_security.py`, then run the check.

In [ ]:
check_mm9_auth(respond, verify)

*Predict first:* does the handshake succeed with Mallory present? And without her?

> **Your prediction:** *(write here, before you run the next cell)*

In [ ]:
with_mallory = dh_chat(mallory=True, authentication=(respond, verify))
without = dh_chat(mallory=False, authentication=(respond, verify))
print(with_mallory, "\n")
print(without)
chat_diagram(with_mallory);

In [ ]:
check_answer("MM9-2c", ...)   # [succeeds with Mallory? (yes / no), succeeds without her?]

**In your own words (2 to 3 sentences):** why can't Mallory just forward Yoshi's response, and why must the response include the public value? What else could replace the pre-shared key in practice (Problem 5)?

> **Your answer:** *(write here)*